# Ensemble of the five-fold models

Notebooks 01–05 give, for each model family, out-of-fold (OOF) probabilities for all 12,260 training images and a fold ensemble for the test set (the mean of its five fold models). Here we combine the families, as notebooks 13, 18 and 19 did with the single fold-0 models.

**What changed.** On fold 0 (27 named-pet photos) every weighting was within noise of every other, so the weights in notebook 19 came from simple rules and the leaderboard judged them. The OOF predictions have 139 named-pet photos, and every image was predicted by models that did not train on it. That is enough to evaluate weightings and per-label thresholds locally, as long as the evaluation is fair:

1. **Candidates.** Every member gets weight 0, 1 or 2. That covers every subset with equal weights, plus doubling some members. With five members this is 211 distinct weightings, and there is no finer search, which would fit noise.
2. **Score of a candidate.** Macro F1 of the weighted mean of the OOF probabilities at per-label thresholds. The thresholds are tuned on the same images, so this score is optimistic, but equally so for every candidate.
3. **Rule.** Take the candidate with the best score, but keep the equal mean of all members unless the best candidate beats it by more than 0.005 (about two named-pet photos). This protects against choosing a weighting that wins by one lucky photo.
4. **Fair check of the whole procedure.** For each fold, apply steps 2–3 to the other four folds only, then score the chosen weights and thresholds on that fold. The pooled result estimates what the procedure is worth on unseen images, and the five choices show how stable it is.

As before, members are combined by averaging probabilities (ensembling), not by a trained second-level model. The stacking in notebook 15 fitted its weights on fold 0 and lost 0.0136 on the leaderboard.

By default the notebook reads only saved results. With `BUILD_SUBMISSION = True` it writes the submission files and a record once, and keeps existing files. Kaggle upload is always manual.

In [ ]:
from __future__ import annotations

import itertools
import json
import math
import os
import sys
from pathlib import Path

import numpy as np

BUILD_SUBMISSION = True  # Writes the CSVs and one record; existing files are kept.
SUBMIT_TO_KAGGLE = False  # No submission API call exists in this notebook.
CV_OUTPUT = os.environ.get('STAGE2_CV_OUTPUT')  # Same folder the model notebooks wrote to.

# CV folder -> name. A model whose notebook has not produced cv.json yet is skipped.
MODELS = {
    'convnext': 'Exp1 ConvNeXt',
    'convnext_custom': 'Exp2 ConvNeXt+block',
    'efficientnetv2s': 'Exp3 EfficientNetV2-S',
    'bit_r50x1': 'Exp5 BiT',
    'hrnet_w32': 'Exp7 HRNet',
}
WEIGHT_CHOICES = (0, 1, 2)
NOISE_MARGIN = 0.005  # About two named-pet photos of macro F1 on the OOF set.
FIXED_THRESHOLD = 0.25
# Best earlier submission, for the cell-by-cell comparison: notebook 19, public 0.98305.
EARLIER_BEST = 'submissions/exp4_weighted_skill_exp1_exp3_exp5_exp7_025.csv'

start = Path(os.environ.get('STAGE2_ROOT', Path.cwd())).resolve()
ROOT = next((path for path in (start, *start.parents) if (path / 'folds.csv').is_file() and (path / 'stage2' / 'cv.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Launch Jupyter from this repository or a subdirectory, or set STAGE2_ROOT.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
CV_ROOT = Path(CV_OUTPUT) if CV_OUTPUT else ROOT / 'experiments' / 'cv'
print('CV results folder:', CV_ROOT)

## Load the members

For every model we read `cv.json`, `oof.csv` and `test_mean.csv`. Each file's hash is checked against the record. All members must use the same fold file and label order, and every training image must have the same held-out fold in all of them. No model is run.

In [ ]:
from stage2.cv import read_cv_summary, sha256, write_json_once, write_matrix
from stage2.ensemble import weighted_mean_probabilities
from stage2.experiment import LABEL_COLUMNS, form_submission, form_submission_per_label, read_csv, score_probabilities, write_submission
from stage2.thresholds import nested_thresholds, per_label_f1, tune_label_thresholds

DATA_DIR = ROOT / 'data'
train_rows = read_csv(DATA_DIR / 'train.csv', ('image_id', *LABEL_COLUMNS))
train_ids = [row['image_id'] for row in train_rows]
targets = np.asarray([[int(row[label]) for label in LABEL_COLUMNS] for row in train_rows])
test_ids = [row['image_id'] for row in read_csv(DATA_DIR / 'test.csv', ('image_id',))]
fold_sha256 = sha256(ROOT / 'folds.csv')

members, missing = {}, []
for key, name in MODELS.items():
    if not (CV_ROOT / key / 'cv.json').is_file():
        missing.append(name)
        continue
    loaded = read_cv_summary(CV_ROOT / key, train_ids, test_ids)
    if loaded['record']['data']['fold_sha256'] != fold_sha256 or loaded['record']['data']['label_columns'] != list(LABEL_COLUMNS):
        raise ValueError(f'{key} used another fold file or label order.')
    members[key] = loaded
if missing:
    print('No five-fold result yet, skipped:', missing)
if len(members) < 2:
    raise RuntimeError('An ensemble needs at least two models with a completed cv.json.')
folds = next(iter(members.values()))['folds']
if any(not np.array_equal(member['folds'], folds) for member in members.values()):
    raise ValueError('The members disagree on the held-out fold of some images.')
names = list(members)
oof = {key: member['oof'] for key, member in members.items()}
test = {key: member['test'] for key, member in members.items()}
print('Members:', [MODELS[key] for key in names])

## The members alone

Pooled OOF macro F1 at 0.25, the fair per-label-threshold score from each model's notebook, and macro AP, which needs no threshold. The last columns count the OOF mistakes on the 139 named-pet photos and the 12,121 non-pet images at 0.25, since these photos decide macro F1.

In [ ]:
named = [LABEL_COLUMNS.index(label) for label in ('Moris', 'Motya', 'Biatrix')]
errors = {}
print(f"{'model':<24}{'F1 @0.25':>10}{'F1 fair':>9}{'AP':>9}{'pets missed':>13}{'pet false alarms':>18}")
for key in names:
    scores = score_probabilities(targets, oof[key], FIXED_THRESHOLD)
    decided = oof[key] >= FIXED_THRESHOLD
    errors[key] = decided[:, named] != targets[:, named].astype(bool)
    missed = int((~decided[:, named] & (targets[:, named] == 1)).sum())
    alarms = int((decided[:, named] & (targets[:, named] == 0)).sum())
    fair = members[key]['record']['per_label_thresholds']['fair_macro_f1']
    print(f"{MODELS[key]:<24}{scores['macro_f1']:>10.5f}{fair:>9.5f}{scores['macro_ap']:>9.5f}{missed:>13}{alarms:>18}")

## Do the members make different mistakes?

An average can only fix a mistake that most members do not share. For every pair of models we count the named-pet OOF cells that both get wrong at 0.25 (the diagonal is a model's own count). The last line counts the cells that every member gets wrong; no weighting can fix those.

In [ ]:
print(f"{'shared errors':<24}" + ''.join(f'{MODELS[key].split()[0]:>8}' for key in names))
for a in names:
    print(f'{MODELS[a]:<24}' + ''.join(f'{int((errors[a] & errors[b]).sum()):>8}' for b in names))
every = np.logical_and.reduce([errors[key] for key in names])
print(f'\nNamed-pet cells wrong in every member: {int(every.sum())}; wrong in at least one: {int(np.logical_or.reduce([errors[key] for key in names]).sum())}')

## Search the weightings, with a fair check

Each candidate gets weights from {0, 1, 2}. Two candidates that differ only by a common factor, such as (1, 1, 0, …) and (2, 2, 0, …), are the same mean, so they are counted once. A candidate is scored on the OOF probabilities with per-label thresholds tuned on the same images, and the rule from the introduction picks one. The table shows the 10 best candidates and the equal mean of all members, each also with its fair (nested-threshold) score and its score at 0.25.

Then the fair check of the whole procedure: four folds choose the weights and thresholds, and the fifth fold scores them. The pooled result is the number to compare with the members' fair scores above. With five members the cell takes well under a minute.

In [ ]:
candidates = sorted({tuple(np.asarray(w) // math.gcd(*w)) for w in itertools.product(WEIGHT_CHOICES, repeat=len(names)) if any(w)})
candidates = [tuple(int(v) for v in weights) for weights in candidates]
EQUAL = tuple([1] * len(names))


def combine(probabilities: dict, weights: tuple, rows=slice(None)) -> np.ndarray:
    active = [index for index, weight in enumerate(weights) if weight > 0]
    if len(active) == 1:
        return probabilities[names[active[0]]][rows]
    return weighted_mean_probabilities([probabilities[names[index]][rows] for index in active], [weights[index] for index in active])


def tuned_score(weights: tuple, rows: np.ndarray) -> float:
    probabilities = combine(oof, weights, rows)
    return float(per_label_f1(targets[rows], probabilities, tune_label_thresholds(targets[rows], probabilities)).mean())


def choose(rows: np.ndarray) -> tuple[tuple, dict]:
    scores = {weights: tuned_score(weights, rows) for weights in candidates}
    best = max(scores, key=lambda weights: (scores[weights], -sum(weights)))
    return (best if scores[best] > scores[EQUAL] + NOISE_MARGIN else EQUAL), scores


everything = np.ones(len(targets), dtype=bool)
chosen, all_scores = choose(everything)
show = sorted(all_scores, key=all_scores.get, reverse=True)[:10]
show += [EQUAL] if EQUAL not in show else []
print(f"{len(candidates)} candidates. Weights in the order {[MODELS[key].split()[0] for key in names]}.")
print(f"{'weights':<22}{'tuned (optimistic)':>20}{'fair thresholds':>17}{'@0.25':>9}")
for weights in show:
    probabilities = combine(oof, weights)
    fair = nested_thresholds(targets, probabilities, folds)['fair_macro_f1']
    mark = ' <- chosen' if weights == chosen else (' (equal mean)' if weights == EQUAL else '')
    print(f"{str(weights):<22}{all_scores[weights]:>20.5f}{fair:>17.5f}{score_probabilities(targets, probabilities, FIXED_THRESHOLD)['macro_f1']:>9.5f}{mark}")

decisions = np.zeros(targets.shape, dtype=bool)
choices_by_fold = {}
for fold in range(5):
    train_part = folds != fold
    weights, _ = choose(train_part)
    thresholds = tune_label_thresholds(targets[train_part], combine(oof, weights, train_part))
    decisions[~train_part] = combine(oof, weights, ~train_part) >= thresholds
    choices_by_fold[fold] = {'weights': weights, 'thresholds': thresholds.round(2).tolist()}
    print(f'held-out fold {fold}: weights {weights}, thresholds {thresholds.round(2).tolist()}')
procedure_fair = per_label_f1(targets, decisions.astype(float), 0.5)
print(f'\nFair macro F1 of the whole procedure (weights and thresholds chosen without the scored fold): {procedure_fair.mean():.5f}')
print('Per label:', dict(zip(LABEL_COLUMNS, procedure_fair.round(4).tolist())))

## The final ensembles on the test set

The weights chosen on all five folds, and the equal mean of all members, are applied to the members' fold-ensemble test probabilities. Each gets the per-label thresholds tuned on its OOF mean. The equal mean is also written at 0.25, so its effect can be separated from the thresholds'. We count the named-pet test cells that change compared with our best earlier submission (notebook 19, `skill` weighting, public 0.98305). One such cell is worth about 0.0007 on the test set.

In [ ]:
named_slice = slice(2, 5)
finals = {'equal_all': EQUAL} | ({'selected': chosen} if chosen != EQUAL else {})
final = {}
for title, weights in finals.items():
    oof_mean = combine(oof, weights)
    tuning = nested_thresholds(targets, oof_mean, folds)
    thresholds = np.array([tuning['thresholds'][label] for label in LABEL_COLUMNS])
    final[title] = {'weights': weights, 'oof': oof_mean, 'test': combine(test, weights), 'tuning': tuning, 'thresholds': thresholds}
    print(f"{title}: weights {dict(zip(names, weights))}; thresholds {dict(zip(LABEL_COLUMNS, thresholds.round(2).tolist()))}")
    print(f"   OOF macro F1 at 0.25 {score_probabilities(targets, oof_mean, FIXED_THRESHOLD)['macro_f1']:.5f}; fair per-label {tuning['fair_macro_f1']:.5f}")

earlier_path = ROOT / EARLIER_BEST
earlier = None
if earlier_path.is_file():
    rows = {row['image_id']: row for row in read_csv(earlier_path, ('image_id', *LABEL_COLUMNS))}
    earlier = np.asarray([[int(rows[image_id][label]) for label in LABEL_COLUMNS] for image_id in test_ids], dtype=bool)
print(f"\n{'test positives':<34}" + ''.join(f'{label:>9}' for label in LABEL_COLUMNS) + f"{'pet cells vs earlier best':>27}")
variants = {'earlier best (public 0.98305)': earlier} if earlier is not None else {}
for title, item in final.items():
    variants[f'{title}, per-label'] = item['test'] >= item['thresholds']
variants['equal_all, 0.25'] = final['equal_all']['test'] >= FIXED_THRESHOLD
for title, decided in variants.items():
    change = '' if earlier is None else f'{int((decided[:, named_slice] != earlier[:, named_slice]).sum()):>27}'
    print(f'{title:<34}' + ''.join(f'{int(decided[:, index].sum()):>9}' for index in range(len(LABEL_COLUMNS))) + change)

## Write the submission files

Up to three CSVs: `cv_ensemble_equal_all_025.csv`, `cv_ensemble_equal_all_perlabel.csv` and, if the rule chose other weights, `cv_ensemble_selected_perlabel.csv`. Each goes with a record and its OOF and test probabilities under `experiments/cv/ensemble/<run-id>/`. The run ID lists the members, so a later run with more members gets new files instead of replacing these. Existing files are kept. Upload by hand and record the public score in the run record and the table below.

In [ ]:
import subprocess

git_commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=ROOT, capture_output=True, text=True).stdout.strip() or None
member_tag = '_'.join(names)
outputs = [('equal_all', '025'), ('equal_all', 'perlabel')] + ([('selected', 'perlabel')] if 'selected' in final else [])
for title, suffix in outputs if BUILD_SUBMISSION else ():
    item = final[title]
    run_id = f'cv_ensemble_{title}_{suffix}_{member_tag}'
    result_dir = CV_ROOT / 'ensemble' / run_id
    submission = ROOT / 'submissions' / f'cv_ensemble_{title}_{suffix}.csv'
    if (result_dir / 'record.json').exists() or submission.exists():
        print('Kept existing', run_id)
        continue
    thresholds = item['thresholds'] if suffix == 'perlabel' else np.full(len(LABEL_COLUMNS), FIXED_THRESHOLD)
    rows = form_submission_per_label(DATA_DIR / 'sample_submission.csv', test_ids, item['test'], thresholds)
    write_matrix(result_dir / 'oof.csv', train_ids, item['oof'], folds)
    write_matrix(result_dir / 'test.csv', test_ids, item['test'])
    write_submission(submission, rows)
    write_json_once(result_dir / 'record.json', {
        'run_id': run_id, 'status': 'completed', 'method': 'weighted mean of five-fold member probabilities, then thresholds',
        'members': {key: {'name': MODELS[key], 'weight': int(weight), 'cv_record_sha256': sha256(CV_ROOT / key / 'cv.json')} for key, weight in zip(names, item['weights'])},
        'weight_choices': list(WEIGHT_CHOICES), 'noise_margin': NOISE_MARGIN, 'thresholds': dict(zip(LABEL_COLUMNS, map(float, thresholds))),
        'oof_macro_f1_at_thresholds': float(per_label_f1(targets, item['oof'], thresholds).mean()),
        'oof_fair_macro_f1_per_label_thresholds': item['tuning']['fair_macro_f1'],
        'procedure_fair_macro_f1': float(procedure_fair.mean()), 'procedure_choices_by_fold': {str(k): {'weights': list(v['weights']), 'thresholds': v['thresholds']} for k, v in choices_by_fold.items()},
        'data': {'fold_file': 'folds.csv', 'fold_sha256': fold_sha256, 'label_columns': list(LABEL_COLUMNS)},
        'submission_csv': str(submission.relative_to(ROOT)), 'positives': {label: sum(row[label] for row in rows) for label in LABEL_COLUMNS},
        'git_commit': git_commit, 'public_leaderboard_score': None,
    })
    print('Wrote', submission.relative_to(ROOT))

## Results and conclusions

_Fill in after the uploads._

| Submission | OOF macro F1 (fair) | Public LB |
|---|---:|---:|
| Earlier best: `skill` weighting of fold-0 models, 0.25 (notebook 19) | n/a (fold 0 only) | 0.98305 |
| Equal mean of the five-fold models, 0.25 | | |
| Equal mean of the five-fold models, per-label thresholds | | |
| Selected weights, per-label thresholds (if different) | | |

Questions to answer here:

1. **Did five-fold training help on its own?** Compare the equal mean at 0.25 with the earlier best. Both use a fixed threshold, so the difference comes from the models.
2. **Did per-label thresholds help?** Compare the two equal-mean rows: the leaderboard difference against the fair OOF gain.
3. **Did the weights matter?** Compare the procedure's fair score with the equal mean's, and check whether the five folds chose the same weights.
4. **Which members carry the ensemble,** and are the cells that every member gets wrong a pattern (small pets, look-alike animals) that a different model or input size might fix?